In [0]:
from datetime import date, datetime
from calendar import monthrange
import json

# ============================================================
# PARAMS
# ============================================================
date_of_data = dbutils.widgets.get("date_of_data")
run_id = dbutils.widgets.get("run_id")

map_reset_checkpoint = {"True": True, "False": False}
reset_checkpoint = map_reset_checkpoint.get(dbutils.widgets.get("reset_checkpoint"), False)

try:
    date_of_data = datetime.strptime(date_of_data, "%Y-%m-%d")
except ValueError:
    raise ValueError(
        f"Uncorrect date: '{date_of_data}'. "
        "Expected format: YYYY-MM-DD."
    )

last_day = monthrange(
    date_of_data.year,
    date_of_data.month
)[1]

date_of_data = date_of_data.replace(day=last_day).date()

In [0]:
# ============================================================
# FROM DATE_OF_DATA GET YEAR AND MONTH
# ============================================================
year = date_of_data.strftime("%Y")
month = date_of_data.strftime("%m")
taxi = ["yellow", "green"]

In [0]:
from pyspark.sql.functions import lit, col, current_timestamp, when, coalesce, sum as spark_sum, cast
from functools import reduce

In [0]:
# ============================================================
# COLUMN NAMES TO LOWERCASE
# ============================================================
def columns_to_lowercase(df):
    return df.toDF(*[c.lower() for c in df.columns])

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    DateType,
    StringType,
    DoubleType,
    IntegerType,
    TimestampType
)

# ============================================================
# SCHEMA FOR EMPTY DATAFRAME
# ============================================================

empty_schema = StructType([
    StructField("date_of_data", DateType(), True),
    StructField("task_id", StringType(), True),
    StructField("taxi_type", StringType(), True),
    StructField("airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("dolocationid", IntegerType(), True),
    StructField("ehail_fee", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("dropoff_datetime", TimestampType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("pulocationid", IntegerType(), True),
    StructField("ratecodeid", IntegerType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("trip_type", StringType(), True),
    StructField("vendorid", IntegerType(), True),
    StructField("loaded_at", TimestampType(), True),
    StructField("rescued_data", StringType(), True),
])

In [0]:
# ============================================================
# COALESCE, BATCH LOADING FUNCTION
# ============================================================
numeric_fill_map = {
    "cbd_congestion_fee": 0.0,
    "congestion_surcharge": 0.0,
    "extra": 0.0,
    "fare_amount": 0.0,
    "improvement_surcharge": 0.0,
    "mta_tax": 0.0,
    "passenger_count": 0,
    "tip_amount": 0.0,
    "tolls_amount": 0.0,
    "total_amount": 0.0,
    "trip_distance": 0.0
}

string_fill_map = {
    "store_and_fwd_flag": "Unknown"
}

required_not_null_columns = [
    "pickup_datetime",
    "dropoff_datetime",
    "pulocationid",
    "dolocationid",
    "vendorid"
]

final_columns = [
    "date_of_data",
    "task_id",
    "taxi_type",
    "airport_fee",
    "cbd_congestion_fee",
    "congestion_surcharge",
    "dolocationid",
    "ehail_fee",
    "extra",
    "fare_amount",
    "improvement_surcharge",
    "pickup_datetime",
    "dropoff_datetime",
    "mta_tax",
    "passenger_count",
    "payment_type",
    "pulocationid",
    "ratecodeid",
    "rescued_data",
    "store_and_fwd_flag",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "trip_distance",
    "trip_type",
    "vendorid",
    "loaded_at"
]

def build_null_condition(columns):
    return reduce(
        lambda left, right: left | right,
        [col(column_name).isNull() for column_name in columns]
    )

def process_silver_batch(batch_df, batch_id):
    taxi_type = batch_df.select("taxi_type").first()["taxi_type"]

    null_condition = build_null_condition(batch_df.columns)

    cleaned_batch = (
        batch_df
        .na.fill(numeric_fill_map)
        .na.fill(string_fill_map)
        .withColumn("loaded_at", current_timestamp())
        .withColumn("payment_type", (
            when(col("payment_type") == 0, "Flex Fare trip")
            .when(col("payment_type") == 1, "Credit card")
            .when(col("payment_type") == 2, "Cash")
            .when(col("payment_type") == 3, "No charge")
            .when(col("payment_type") == 4, "Dispute")
            .when(col("payment_type") == 5, "Unknown")
            .when(col("payment_type") == 6, "Voided trip")
            .otherwise("Unknown")
        ))
        .withColumn("ratecode", (
            when(col("ratecodeid") == 1, "Standard rate")
            .when(col("ratecodeid") == 2, "JFK")
            .when(col("ratecodeid") == 3, "Newark")
            .when(col("ratecodeid") == 4, "Nassau or Westchester")
            .when(col("ratecodeid") == 5, "Negotiated fare")
            .when(col("ratecodeid") == 6, "Group ride")
            .otherwise("Unknown")
        ))
    )

    if taxi_type == "yellow":
        cleaned_batch = (
            cleaned_batch
            .withColumn("airport_fee", coalesce(col("airport_fee"), lit(0.0)))
            .withColumn("ehail_fee", lit(0.0))
            .withColumn("trip_type", lit("Unknown"))
            .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
            .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
        ).dropna(subset=required_not_null_columns)
    else:
        cleaned_batch = (
            cleaned_batch
            .withColumn("airport_fee", lit(0.0))
            .withColumn("ehail_fee", coalesce(col("ehail_fee"), lit(0.0)))
            .withColumn(
                "trip_type",
                when(col("trip_type") == 1, "Street-hail")
                .when(col("trip_type") == 2, "Dispatch")
                .otherwise("Unknown")
            )
            .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")
            .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
        ).dropna(subset=required_not_null_columns)

    cleaned_batch = cleaned_batch.select(*final_columns)

    if batch_id == 0:
        (
            cleaned_batch.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere",
                    f"date_of_data = '{date_of_data}' "
                    f"and taxi_type = '{taxi_type}'")
            .option("mergeSchema", "true")
            .saveAsTable(destination_path)
        )
        first_batch = False
    else:
        (
            cleaned_batch.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(destination_path)
        )

In [0]:
# ============================================================
# ETL/ELT
# ============================================================

for t in taxi:
    base_volume_path = "/Volumes/bronze/dbo/ny_taxi/"
    partition = f"{t}/{year}/{month}/"

    errors_path = base_volume_path + "errors/" + partition
    checkpoint_path = base_volume_path + "checkpoint/" + partition
    source_path = base_volume_path + partition
    destination_path = "silver.dbo.ny_taxi"

    cloudfile = dict(ignoreCorruptFiles=True)
    cloudfile["cloudFiles.format"] = "parquet"
    cloudfile["pathGlobFilter"] = "*.parquet"
    cloudfile["cloudFiles.inferColumnTypes"] = "true"
    cloudfile["cloudFiles.schemaLocation"] = checkpoint_path

    if reset_checkpoint:
        dbutils.fs.rm(checkpoint_path, True)

    # ========================================================
    # TRY TO LOAD SOURCE
    # ========================================================

    try:
        df_raw_files = (
            spark.readStream
            .format("cloudFiles")
            .options(**cloudfile)
            .option("badRecordsPath", errors_path)
            .load(source_path)
        )

    except Exception as e:
        df_raw_files = spark.createDataFrame(
            [],
            empty_schema
        )

        continue

    # ========================================================
    # GENERIC TRANSFORMATIONS
    # ========================================================

    df_raw_files = columns_to_lowercase(df_raw_files)

    df_raw_files = (
        df_raw_files
        .withColumn("date_of_data", lit(date_of_data))
        .withColumn("task_id", lit(run_id))
        .withColumn("taxi_type", lit(t))
        .withColumnRenamed("_rescued_data", "rescued_data")
    )

    # ========================================================
    # STREAMING
    # ========================================================

    query = (
        df_raw_files
        .writeStream
        .foreachBatch(process_silver_batch)
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start()
    )

    query.awaitTermination()